# Advanced Tutorial Problems with Solutions
## `OrderedDict` vs Plain `dict`

This notebook follows the **tutorial style** of the supplied notebook:

- introduce one question at a time,
- inspect behavior with small examples,
- explain what the result means,
- build a solution in logical steps,
- test the solution,
- and only then generalize it.

The topic stays focused on the same core comparison:

- insertion order,
- reverse traversal,
- first/last removal,
- moving keys,
- order-sensitive comparison,
- and performance-related design decisions.

The problems here are intentionally different from the previous advanced notebook.


### A note about Python versions

The supplied notebook discusses Python 3.6/3.7-era behavior.

This notebook keeps the same conceptual comparison, but the code is written to run on a modern Python 3 environment.

The main design question remains useful:

> If a plain dictionary preserves insertion order, when do we still gain something from `OrderedDict`?


In [1]:
from collections import OrderedDict
from timeit import repeat
from statistics import median
import random


We will use `OrderedDict` as both:

1. a real data structure, and
2. a useful behavioral reference when we try to reproduce some ordered operations with a plain dictionary.


# Problem 1 — Build a "touch ordered" score table

Suppose we have a score table.

The order should mean:

> "least recently changed" → "most recently changed"

This is slightly different from ordinary insertion order.

We will first investigate what a normal update does.


In [2]:
scores = {
    "alice": 10,
    "bob": 20,
    "carol": 30
}

print(scores)


{'alice': 10, 'bob': 20, 'carol': 30}


Now let's update `bob`.

A natural first guess might be that because `bob` changed most recently, it should move to the end.


In [3]:
scores["bob"] = 25
print(scores)


{'alice': 10, 'bob': 25, 'carol': 30}


But assigning a new value to an existing key does **not** create a new insertion position.

So our order is still:

```text
alice, bob, carol
```

If we want the order to represent "most recently changed", we need to explicitly move the touched key.


### Step 1 — Pop the existing key

If the key already exists, we can remove it first.


In [4]:
scores = {
    "alice": 10,
    "bob": 20,
    "carol": 30
}

old_value = scores.pop("bob")
print(old_value)
print(scores)


20
{'alice': 10, 'carol': 30}


### Step 2 — Insert it again

A new insertion occurs at the end.


In [5]:
scores["bob"] = 25
print(scores)


{'alice': 10, 'carol': 30, 'bob': 25}


That gives us the behavior we want.

Now let's package this into a function.


In [6]:
def touch_set(d, key, value):
    if key in d:
        d.pop(key)

    d[key] = value


In [7]:
scores = {
    "alice": 10,
    "bob": 20,
    "carol": 30
}

touch_set(scores, "alice", 15)

print(scores)


{'bob': 20, 'carol': 30, 'alice': 15}


The result should now place `alice` at the end because `alice` was the most recently changed entry.


In [8]:
assert list(scores.items()) == [
    ("bob", 20),
    ("carol", 30),
    ("alice", 15)
]


### Step 3 — What about a brand-new key?

A new key is already inserted at the end, so no special handling is needed.


In [9]:
touch_set(scores, "dave", 40)
print(scores)


{'bob': 20, 'carol': 30, 'alice': 15, 'dave': 40}


### Solution summary

The important idea is:

```python
if key in d:
    d.pop(key)
d[key] = value
```

This converts ordinary insertion order into **last-touch order**.


# Problem 2 — Build non-destructive "peek first" and "peek last"

The supplied notebook spends time on first/last removal.

Before removing anything, it is useful to solve a simpler problem:

> How can we inspect the first or last item without modifying the dictionary?

We will build both operations carefully.


In [10]:
d = {
    "a": 100,
    "b": 200,
    "c": 300,
    "d": 400
}

print(d)


{'a': 100, 'b': 200, 'c': 300, 'd': 400}


### Step 1 — Find the first key

An iterator over a dictionary yields keys in iteration order.

So the first key is:


In [11]:
first_key = next(iter(d))

print(first_key)
print(d[first_key])


a
100


This gives us enough information to return the first item.


In [12]:
def peek_first(d):
    if not d:
        raise KeyError("dictionary is empty")

    key = next(iter(d))
    return key, d[key]


In [13]:
print(peek_first(d))


('a', 100)


### Step 2 — Find the last key

In a modern Python environment, dictionaries support reverse iteration.

So:


In [14]:
last_key = next(reversed(d))

print(last_key)
print(d[last_key])


d
400


Now we can write the matching helper.


In [15]:
def peek_last(d):
    if not d:
        raise KeyError("dictionary is empty")

    key = next(reversed(d))
    return key, d[key]


In [16]:
print(peek_last(d))


('d', 400)


### Step 3 — Verify that peeking is non-destructive


In [17]:
before = list(d.items())

_ = peek_first(d)
_ = peek_last(d)

after = list(d.items())

print(before)
print(after)

assert before == after


[('a', 100), ('b', 200), ('c', 300), ('d', 400)]
[('a', 100), ('b', 200), ('c', 300), ('d', 400)]


### Step 4 — Empty dictionary behavior

We should make failure explicit rather than allowing a confusing `StopIteration`.


In [18]:
for fn in (peek_first, peek_last):
    try:
        fn({})
    except KeyError as exc:
        print(fn.__name__, "->", exc)


peek_first -> 'dictionary is empty'
peek_last -> 'dictionary is empty'


# Problem 3 — Build a bounded "recent changes" registry

Now we will combine two ideas:

1. touched keys move to the end,
2. the oldest key is at the beginning.

We want a registry with a maximum capacity.

Whenever a new change makes the registry too large, we will evict the oldest entry.


Let's begin with a small example.

We will keep only three entries.


In [19]:
recent = {}

touch_set(recent, "a", 10)
touch_set(recent, "b", 20)
touch_set(recent, "c", 30)

print(recent)


{'a': 10, 'b': 20, 'c': 30}


If we now touch `a`, it should become the newest entry.


In [20]:
touch_set(recent, "a", 11)
print(recent)


{'b': 20, 'c': 30, 'a': 11}


So the oldest entry is now `b`.

Let's retrieve that key.


In [21]:
oldest_key = next(iter(recent))
print(oldest_key)


b


### Step 1 — Add a fourth entry


In [22]:
touch_set(recent, "d", 40)
print(recent)


{'b': 20, 'c': 30, 'a': 11, 'd': 40}


We now have four entries, but our capacity is three.

So we need to remove the first item.


In [23]:
oldest_key = next(iter(recent))
evicted = (oldest_key, recent.pop(oldest_key))

print("evicted:", evicted)
print("remaining:", recent)


evicted: ('b', 20)
remaining: {'c': 30, 'a': 11, 'd': 40}


### Step 2 — Generalize the logic


In [24]:
def record_change(d, key, value, *, capacity):
    if capacity < 1:
        raise ValueError("capacity must be at least 1")

    touch_set(d, key, value)

    if len(d) > capacity:
        oldest_key = next(iter(d))
        return oldest_key, d.pop(oldest_key)

    return None


Let's test a sequence of changes.


In [25]:
recent = {}

print(record_change(recent, "a", 1, capacity=3))
print(record_change(recent, "b", 2, capacity=3))
print(record_change(recent, "c", 3, capacity=3))
print(record_change(recent, "a", 100, capacity=3))
print(record_change(recent, "d", 4, capacity=3))

print(recent)


None
None
None
None
('b', 2)
{'c': 3, 'a': 100, 'd': 4}


The expected order is:

```text
c, a, d
```

because `b` was the least recently changed item when `d` was added.


In [26]:
assert list(recent.items()) == [
    ("c", 3),
    ("a", 100),
    ("d", 4)
]


# Problem 4 — Create a "rotate left" operation for a dictionary

Suppose we have:

```text
a b c d e
```

and want to rotate left by one position:

```text
b c d e a
```

This is not exactly `move_to_end`, but it can be built from the same idea.

We will solve the one-step case first.


In [27]:
d = dict(a=1, b=2, c=3, d=4, e=5)
print(d)


{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}


### Step 1 — Find the first key


In [28]:
first_key = next(iter(d))
print(first_key)


a


### Step 2 — Remove it and insert it again


In [29]:
d[first_key] = d.pop(first_key)
print(d)


{'b': 2, 'c': 3, 'd': 4, 'e': 5, 'a': 1}


That is a one-position left rotation.

Now let's make it reusable.


In [30]:
def rotate_left_once(d):
    if not d:
        return

    first_key = next(iter(d))
    d[first_key] = d.pop(first_key)


### Step 3 — Rotate multiple times


In [31]:
d = dict(a=1, b=2, c=3, d=4, e=5)

rotate_left_once(d)
rotate_left_once(d)

print(d)


{'c': 3, 'd': 4, 'e': 5, 'a': 1, 'b': 2}


After two rotations, we expect:

```text
c d e a b
```


In [32]:
assert list(d) == ["c", "d", "e", "a", "b"]


### Step 4 — Generalize to `rotate_left(d, n)`

There is one important detail:

Rotating a dictionary of length 5 by 7 positions is the same as rotating by 2 positions.

So we can reduce the work using modulo.


In [33]:
def rotate_left(d, n=1):
    if not d:
        return

    n %= len(d)

    for _ in range(n):
        first_key = next(iter(d))
        d[first_key] = d.pop(first_key)


In [34]:
d = dict(a=1, b=2, c=3, d=4, e=5)

rotate_left(d, 7)

print(d)


{'c': 3, 'd': 4, 'e': 5, 'a': 1, 'b': 2}


In [35]:
assert list(d) == ["c", "d", "e", "a", "b"]


# Problem 5 — Create a "rotate right" operation

Now we want the opposite transformation:

```text
a b c d e
```

becomes:

```text
e a b c d
```

With an `OrderedDict`, moving a key to the beginning is directly supported.

Let's see that first.


In [36]:
od = OrderedDict(a=1, b=2, c=3, d=4, e=5)

od.move_to_end("e", last=False)

print(od)


OrderedDict({'e': 5, 'a': 1, 'b': 2, 'c': 3, 'd': 4})


Now let's reproduce the same effect with a plain dictionary.

One simple strategy is to rebuild the order.


### Step 1 — Extract the last item


In [37]:
d = dict(a=1, b=2, c=3, d=4, e=5)

last_key = next(reversed(d))
last_value = d[last_key]

print(last_key, last_value)


e 5


### Step 2 — Build the desired item sequence


In [38]:
items = [(last_key, last_value)]

for key, value in d.items():
    if key != last_key:
        items.append((key, value))

print(items)


[('e', 5), ('a', 1), ('b', 2), ('c', 3), ('d', 4)]


### Step 3 — Replace the dictionary contents


In [39]:
d.clear()
d.update(items)

print(d)


{'e': 5, 'a': 1, 'b': 2, 'c': 3, 'd': 4}


Now package that as a helper.


In [40]:
def move_to_front_rebuild(d, key):
    if key not in d:
        raise KeyError(key)

    value = d[key]

    items = [(key, value)]
    items.extend(
        (k, v)
        for k, v in d.items()
        if k != key
    )

    d.clear()
    d.update(items)


Using that helper, a right rotation is just "move the last key to the front".


In [41]:
def rotate_right_once(d):
    if not d:
        return

    last_key = next(reversed(d))
    move_to_front_rebuild(d, last_key)


In [42]:
d = dict(a=1, b=2, c=3, d=4, e=5)

rotate_right_once(d)

print(d)


{'e': 5, 'a': 1, 'b': 2, 'c': 3, 'd': 4}


In [43]:
assert list(d) == ["e", "a", "b", "c", "d"]


### Step 4 — Multiple right rotations


In [44]:
def rotate_right(d, n=1):
    if not d:
        return

    n %= len(d)

    for _ in range(n):
        rotate_right_once(d)


In [45]:
d = dict(a=1, b=2, c=3, d=4, e=5)

rotate_right(d, 2)

print(d)


{'d': 4, 'e': 5, 'a': 1, 'b': 2, 'c': 3}


In [46]:
assert list(d) == ["d", "e", "a", "b", "c"]


# Problem 6 — Normalize a dictionary to a required key order

Suppose an application expects configuration keys in this order:

```text
host, port, user, password
```

But a dictionary arrives in some other order.

We want to reorder it without changing the key/value content.


In [47]:
config = {
    "password": "secret",
    "host": "localhost",
    "user": "admin",
    "port": 5432
}

print(config)


{'password': 'secret', 'host': 'localhost', 'user': 'admin', 'port': 5432}


We will solve this in two parts:

1. validate that the required keys exist,
2. rebuild the dictionary in the required order.


In [48]:
required_order = ["host", "port", "user", "password"]

missing = [
    key
    for key in required_order
    if key not in config
]

print(missing)


[]


There are no missing keys, so we can construct the desired sequence.


In [49]:
ordered_items = [
    (key, config[key])
    for key in required_order
]

print(ordered_items)


[('host', 'localhost'), ('port', 5432), ('user', 'admin'), ('password', 'secret')]


Now rebuild the dictionary.


In [50]:
config.clear()
config.update(ordered_items)

print(config)


{'host': 'localhost', 'port': 5432, 'user': 'admin', 'password': 'secret'}


### General solution

We will also preserve any extra keys after the required ones.


In [51]:
def normalize_key_order(d, required_order):
    missing = [
        key
        for key in required_order
        if key not in d
    ]

    if missing:
        raise KeyError(f"missing keys: {missing}")

    required_set = set(required_order)

    items = [
        (key, d[key])
        for key in required_order
    ]

    items.extend(
        (key, value)
        for key, value in d.items()
        if key not in required_set
    )

    d.clear()
    d.update(items)


In [52]:
config = {
    "debug": True,
    "password": "secret",
    "host": "localhost",
    "user": "admin",
    "port": 5432,
    "timeout": 30
}

normalize_key_order(
    config,
    ["host", "port", "user", "password"]
)

print(config)


{'host': 'localhost', 'port': 5432, 'user': 'admin', 'password': 'secret', 'debug': True, 'timeout': 30}


In [53]:
assert list(config) == [
    "host",
    "port",
    "user",
    "password",
    "debug",
    "timeout"
]


# Problem 7 — Detect a pure order change

Ordinary dictionary equality ignores insertion order.

That is usually exactly what we want.

But sometimes order itself matters—for example, when comparing two serialized configurations or two processing pipelines.

We want to classify two mappings as:

- identical in content and order,
- same content but different order,
- different content.


In [54]:
d1 = {
    "a": 10,
    "b": 20,
    "c": 30
}

d2 = {
    "b": 20,
    "c": 30,
    "a": 10
}

print(d1 == d2)


True


The dictionaries compare equal because they contain the same key/value pairs.

Now compare their item sequences.


In [55]:
print(list(d1.items()))
print(list(d2.items()))

print(list(d1.items()) == list(d2.items()))


[('a', 10), ('b', 20), ('c', 30)]
[('b', 20), ('c', 30), ('a', 10)]
False


That gives us the two tests we need:

1. ordinary equality for content,
2. item-sequence equality for order.


In [56]:
def classify_mapping_change(old, new):
    if list(old.items()) == list(new.items()):
        return "identical"

    if old == new:
        return "order-only"

    return "content-change"


In [57]:
same = {
    "a": 10,
    "b": 20,
    "c": 30
}

reordered = {
    "b": 20,
    "c": 30,
    "a": 10
}

changed = {
    "a": 10,
    "b": 999,
    "c": 30
}

print(classify_mapping_change(d1, same))
print(classify_mapping_change(d1, reordered))
print(classify_mapping_change(d1, changed))


identical
order-only
content-change


### A more streaming-style order check

Instead of materializing both full item lists, we can compare corresponding items one by one.


In [58]:
def same_order_and_content(d1, d2):
    if len(d1) != len(d2):
        return False

    for item1, item2 in zip(d1.items(), d2.items()):
        if item1 != item2:
            return False

    return True


In [59]:
print(same_order_and_content(d1, same))
print(same_order_and_content(d1, reordered))


True
False


# Problem 8 — Create a deterministic "dictionary fingerprint"

Suppose we need a simple textual fingerprint of a mapping.

We want two modes:

1. **content mode** — ignore key order,
2. **ordered mode** — include key order.

This problem is useful because it forces us to make the order semantics explicit.


In [60]:
d1 = {"b": 2, "a": 1, "c": 3}
d2 = {"a": 1, "b": 2, "c": 3}

print(d1 == d2)


True


### Step 1 — Content-only fingerprint

If order should not matter, sort the items before turning them into text.


In [61]:
content_fingerprint_1 = repr(sorted(d1.items()))
content_fingerprint_2 = repr(sorted(d2.items()))

print(content_fingerprint_1)
print(content_fingerprint_2)

assert content_fingerprint_1 == content_fingerprint_2


[('a', 1), ('b', 2), ('c', 3)]
[('a', 1), ('b', 2), ('c', 3)]


### Step 2 — Order-sensitive fingerprint

If order matters, preserve the item sequence exactly as stored.


In [62]:
ordered_fingerprint_1 = repr(list(d1.items()))
ordered_fingerprint_2 = repr(list(d2.items()))

print(ordered_fingerprint_1)
print(ordered_fingerprint_2)

assert ordered_fingerprint_1 != ordered_fingerprint_2


[('b', 2), ('a', 1), ('c', 3)]
[('a', 1), ('b', 2), ('c', 3)]


### Step 3 — Generalize


In [63]:
def mapping_fingerprint(d, *, order_sensitive=False):
    if order_sensitive:
        items = list(d.items())
    else:
        items = sorted(d.items())

    return repr(items)


In [64]:
print(mapping_fingerprint(d1))
print(mapping_fingerprint(d2))

print(mapping_fingerprint(d1, order_sensitive=True))
print(mapping_fingerprint(d2, order_sensitive=True))


[('a', 1), ('b', 2), ('c', 3)]
[('a', 1), ('b', 2), ('c', 3)]
[('b', 2), ('a', 1), ('c', 3)]
[('a', 1), ('b', 2), ('c', 3)]


This is a deliberately simple fingerprint, not a cryptographic hash.

The important lesson is the distinction between:

- dictionary content,
- and dictionary content **plus order**.


# Problem 9 — Merge two mappings with "last touched" semantics

Normal `dict.update()` overwrites duplicate values, but it does not move an existing key to a new position.

Let's verify that first.


In [65]:
left = {
    "a": 1,
    "b": 2,
    "c": 3
}

right = {
    "b": 200,
    "d": 4
}

result = left.copy()
result.update(right)

print(result)


{'a': 1, 'b': 200, 'c': 3, 'd': 4}


Notice that `b` is still in its original position.

Now suppose our order is meant to represent the time of the most recent update.

Then every key processed from `right` should become "new".


### Step 1 — Process one item manually


In [66]:
result = left.copy()

key = "b"
value = 200

if key in result:
    result.pop(key)

result[key] = value

print(result)


{'a': 1, 'c': 3, 'b': 200}


Now `b` moved to the end.

Let's process `d`.


In [67]:
key = "d"
value = 4

if key in result:
    result.pop(key)

result[key] = value

print(result)


{'a': 1, 'c': 3, 'b': 200, 'd': 4}


### Step 2 — Generalize the merge


In [68]:
def merge_by_touch(left, right):
    result = dict(left)

    for key, value in right.items():
        if key in result:
            result.pop(key)

        result[key] = value

    return result


In [69]:
left = {
    "a": 1,
    "b": 2,
    "c": 3
}

right = {
    "b": 200,
    "d": 4,
    "a": 100,
    "e": 5
}

merged = merge_by_touch(left, right)

print(merged)


{'c': 3, 'b': 200, 'd': 4, 'a': 100, 'e': 5}


Let's reason through the final order.

- `c` was never touched by `right`, so it remains first among old untouched entries.
- `b` was touched first.
- `d` was added next.
- `a` was touched after that.
- `e` was added last.

So we expect:


In [70]:
assert list(merged.items()) == [
    ("c", 3),
    ("b", 200),
    ("d", 4),
    ("a", 100),
    ("e", 5)
]


# Problem 10 — Build an MRU list with `OrderedDict`

Now let's solve a problem where `OrderedDict` gives us a particularly natural API.

We will create an MRU list:

> Most Recently Used items are kept at the end.

When an item is accessed, it moves to the end.

When the structure becomes too large, we remove the least recently used item from the beginning.


In [71]:
recent = OrderedDict()

recent["A"] = "alpha"
recent["B"] = "beta"
recent["C"] = "gamma"

print(recent)


OrderedDict({'A': 'alpha', 'B': 'beta', 'C': 'gamma'})


### Step 1 — Touch an existing item

`move_to_end` directly expresses the operation we want.


In [72]:
recent.move_to_end("A")
print(recent)


OrderedDict({'B': 'beta', 'C': 'gamma', 'A': 'alpha'})


Now the order is:

```text
B, C, A
```

so `B` is least recent and `A` is most recent.


### Step 2 — Add one more item


In [73]:
recent["D"] = "delta"
print(recent)


OrderedDict({'B': 'beta', 'C': 'gamma', 'A': 'alpha', 'D': 'delta'})


### Step 3 — Remove the least recent item

This is exactly what `popitem(last=False)` does.


In [74]:
evicted = recent.popitem(last=False)

print("evicted:", evicted)
print("remaining:", recent)


evicted: ('B', 'beta')
remaining: OrderedDict({'C': 'gamma', 'A': 'alpha', 'D': 'delta'})


### Step 4 — Package the behavior into a class


In [75]:
class MRURegistry:
    def __init__(self, capacity):
        if capacity < 1:
            raise ValueError("capacity must be at least 1")

        self.capacity = capacity
        self._data = OrderedDict()

    def set(self, key, value):
        if key in self._data:
            self._data[key] = value
            self._data.move_to_end(key)
        else:
            self._data[key] = value

        if len(self._data) > self.capacity:
            return self._data.popitem(last=False)

        return None

    def get(self, key):
        value = self._data[key]
        self._data.move_to_end(key)
        return value

    def snapshot(self):
        return list(self._data.items())


In [76]:
registry = MRURegistry(3)

registry.set("A", 1)
registry.set("B", 2)
registry.set("C", 3)

print(registry.snapshot())

registry.get("A")
print(registry.snapshot())

evicted = registry.set("D", 4)

print("evicted:", evicted)
print("final:", registry.snapshot())


[('A', 1), ('B', 2), ('C', 3)]
[('B', 2), ('C', 3), ('A', 1)]
evicted: ('B', 2)
final: [('C', 3), ('A', 1), ('D', 4)]


In [77]:
assert evicted == ("B", 2)
assert registry.snapshot() == [
    ("C", 3),
    ("A", 1),
    ("D", 4)
]


### Why `OrderedDict` fits this problem well

A plain dictionary can emulate this behavior.

But here the application is explicitly about reordering.

Using `OrderedDict` makes the intended operations very visible:

```python
move_to_end(...)
popitem(last=False)
```


# Problem 11 — Build a two-ended job queue

We will use `OrderedDict` as a queue-like structure where every job has a unique key.

We want to support:

- add a normal job to the end,
- add an urgent job to the beginning,
- process the oldest/front job,
- process the newest/back job.


Let's start with three jobs.


In [78]:
jobs = OrderedDict([
    ("job-1", "compile"),
    ("job-2", "test"),
    ("job-3", "package")
])

print(jobs)


OrderedDict({'job-1': 'compile', 'job-2': 'test', 'job-3': 'package'})


### Step 1 — Add an urgent job

We can insert it normally and then move it to the beginning.


In [79]:
jobs["urgent-1"] = "security patch"
jobs.move_to_end("urgent-1", last=False)

print(jobs)


OrderedDict({'urgent-1': 'security patch', 'job-1': 'compile', 'job-2': 'test', 'job-3': 'package'})


### Step 2 — Process from the front


In [80]:
processed_front = jobs.popitem(last=False)

print(processed_front)
print(jobs)


('urgent-1', 'security patch')
OrderedDict({'job-1': 'compile', 'job-2': 'test', 'job-3': 'package'})


### Step 3 — Process from the back


In [81]:
processed_back = jobs.popitem(last=True)

print(processed_back)
print(jobs)


('job-3', 'package')
OrderedDict({'job-1': 'compile', 'job-2': 'test'})


### Step 4 — Wrap the behavior in a class


In [82]:
class JobQueue:
    def __init__(self):
        self._jobs = OrderedDict()

    def add(self, job_id, payload):
        self._jobs[job_id] = payload

    def add_urgent(self, job_id, payload):
        self._jobs[job_id] = payload
        self._jobs.move_to_end(job_id, last=False)

    def pop_front(self):
        return self._jobs.popitem(last=False)

    def pop_back(self):
        return self._jobs.popitem(last=True)

    def snapshot(self):
        return list(self._jobs.items())


In [83]:
queue = JobQueue()

queue.add("j1", "compile")
queue.add("j2", "test")
queue.add_urgent("hotfix", "patch production")
queue.add("j3", "deploy")

print(queue.snapshot())


[('hotfix', 'patch production'), ('j1', 'compile'), ('j2', 'test'), ('j3', 'deploy')]


In [84]:
assert queue.snapshot() == [
    ("hotfix", "patch production"),
    ("j1", "compile"),
    ("j2", "test"),
    ("j3", "deploy")
]


# Problem 12 — Compare two reorder-heavy implementations

The supplied notebook includes timing comparisons.

We will do something similar, but for a workload dominated by moving keys to the beginning.

This is a good case for comparing:

- `OrderedDict.move_to_end(key, last=False)`
- a plain-dictionary rebuild.


First, let's define a small plain-dictionary operation.


In [85]:
def plain_move_front(d, key):
    value = d[key]

    items = [(key, value)]
    items.extend(
        (k, v)
        for k, v in d.items()
        if k != key
    )

    d.clear()
    d.update(items)


Now define one benchmark workload for each mapping type.

Each workload:

1. creates a mapping,
2. repeatedly moves different keys to the front.


In [86]:
def workload_plain(n=1_000, moves=200):
    d = {i: i for i in range(n)}

    for i in range(moves):
        key = (i * 17) % n
        plain_move_front(d, key)

    return d


def workload_ordered(n=1_000, moves=200):
    d = OrderedDict((i, i) for i in range(n))

    for i in range(moves):
        key = (i * 17) % n
        d.move_to_end(key, last=False)

    return d


Before timing, we should verify that both workloads produce the same final order.


In [87]:
plain_result = workload_plain(100, 30)
ordered_result = workload_ordered(100, 30)

print(list(plain_result.items())[:10])
print(list(ordered_result.items())[:10])

assert list(plain_result.items()) == list(ordered_result.items())


[(93, 93), (76, 76), (59, 59), (42, 42), (25, 25), (8, 8), (91, 91), (74, 74), (57, 57), (40, 40)]
[(93, 93), (76, 76), (59, 59), (42, 42), (25, 25), (8, 8), (91, 91), (74, 74), (57, 57), (40, 40)]


Now we can benchmark.

We will use repeated timings and compare medians rather than trusting one measurement.


In [88]:
plain_times = repeat(
    "workload_plain(1_000, 200)",
    globals=globals(),
    number=3,
    repeat=5
)

ordered_times = repeat(
    "workload_ordered(1_000, 200)",
    globals=globals(),
    number=3,
    repeat=5
)

print("plain median  :", median(plain_times))
print("ordered median:", median(ordered_times))


plain median  : 0.1256145006045699
ordered median: 0.000829700380563736


The exact numbers depend on the Python version and machine.

The important design lesson is that a workload with frequent reordering is structurally different from a workload that merely needs insertion-order preservation.


# Problem 13 — Randomly test plain-dict rotation against a reference model

When we write custom order manipulation code, a few hand-written examples are not always enough.

We can test many cases automatically.

For left rotation, a simple list of keys can act as our reference model.


Let's start with one example.


In [89]:
keys = ["a", "b", "c", "d", "e"]

expected = keys[2:] + keys[:2]

print(expected)


['c', 'd', 'e', 'a', 'b']


Now compare that with our dictionary operation.


In [90]:
d = {key: key.upper() for key in keys}

rotate_left(d, 2)

print(list(d))
print(expected)

assert list(d) == expected


['c', 'd', 'e', 'a', 'b']
['c', 'd', 'e', 'a', 'b']


### Step 1 — Generate many random tests


In [91]:
def randomized_rotate_left_test(seed=0, trials=200):
    rng = random.Random(seed)

    for _ in range(trials):
        size = rng.randint(1, 30)
        keys = list(range(size))
        rng.shuffle(keys)

        d = {key: key * 10 for key in keys}

        n = rng.randint(0, 100)

        expected_keys = keys[n % size:] + keys[:n % size]

        rotate_left(d, n)

        assert list(d) == expected_keys

    return True


In [92]:
print(randomized_rotate_left_test())


True


### Step 2 — Repeat with several seeds

Different seeds generate different collections of test cases.


In [93]:
for seed in range(10):
    assert randomized_rotate_left_test(
        seed=seed,
        trials=100
    )

print("all randomized tests passed")


all randomized tests passed


This is a powerful general pattern:

> When order manipulation becomes complicated, compare the result with a simpler reference model.


# Problem 14 — Capstone: an ordered command history

We will finish with a larger tutorial problem.

We want a command history where:

- every command has a unique ID,
- adding a command places it at the end,
- re-running a command makes it most recent,
- pinning a command moves it to the beginning,
- oldest/newest commands can be inspected,
- oldest/newest commands can be removed,
- two histories can be compared with or without order sensitivity.


Because this structure is explicitly reorder-heavy, we will use `OrderedDict`.


### Step 1 — Create the container


In [94]:
class CommandHistory:
    def __init__(self):
        self._commands = OrderedDict()


### Step 2 — Add commands

If an existing command ID is added again, we will update its text and make it most recent.


In [95]:
class CommandHistory:
    def __init__(self):
        self._commands = OrderedDict()

    def add(self, command_id, text):
        self._commands[command_id] = text
        self._commands.move_to_end(command_id)


Let's try it.


In [96]:
history = CommandHistory()

history.add("c1", "load data")
history.add("c2", "clean data")
history.add("c3", "train model")

print(history._commands)


OrderedDict({'c1': 'load data', 'c2': 'clean data', 'c3': 'train model'})


Now re-run `c1`.


In [97]:
history.add("c1", "load data --refresh")

print(history._commands)


OrderedDict({'c2': 'clean data', 'c3': 'train model', 'c1': 'load data --refresh'})


The key `c1` should now be at the end.


In [98]:
assert list(history._commands) == [
    "c2",
    "c3",
    "c1"
]


### Step 3 — Add pinning

Pinning should move a command to the beginning.


In [99]:
class CommandHistory:
    def __init__(self):
        self._commands = OrderedDict()

    def add(self, command_id, text):
        self._commands[command_id] = text
        self._commands.move_to_end(command_id)

    def pin(self, command_id):
        self._commands.move_to_end(
            command_id,
            last=False
        )


In [100]:
history = CommandHistory()

history.add("c1", "load data")
history.add("c2", "clean data")
history.add("c3", "train model")

history.pin("c3")

print(history._commands)


OrderedDict({'c3': 'train model', 'c1': 'load data', 'c2': 'clean data'})


### Step 4 — Add peek operations


In [101]:
class CommandHistory:
    def __init__(self):
        self._commands = OrderedDict()

    def add(self, command_id, text):
        self._commands[command_id] = text
        self._commands.move_to_end(command_id)

    def pin(self, command_id):
        self._commands.move_to_end(
            command_id,
            last=False
        )

    def oldest(self):
        if not self._commands:
            raise KeyError("history is empty")

        key = next(iter(self._commands))
        return key, self._commands[key]

    def newest(self):
        if not self._commands:
            raise KeyError("history is empty")

        key = next(reversed(self._commands))
        return key, self._commands[key]


In [102]:
history = CommandHistory()

history.add("c1", "load data")
history.add("c2", "clean data")
history.add("c3", "train model")

print("oldest:", history.oldest())
print("newest:", history.newest())


oldest: ('c1', 'load data')
newest: ('c3', 'train model')


### Step 5 — Add removal from both ends


In [103]:
class CommandHistory:
    def __init__(self):
        self._commands = OrderedDict()

    def add(self, command_id, text):
        self._commands[command_id] = text
        self._commands.move_to_end(command_id)

    def pin(self, command_id):
        self._commands.move_to_end(
            command_id,
            last=False
        )

    def oldest(self):
        if not self._commands:
            raise KeyError("history is empty")

        key = next(iter(self._commands))
        return key, self._commands[key]

    def newest(self):
        if not self._commands:
            raise KeyError("history is empty")

        key = next(reversed(self._commands))
        return key, self._commands[key]

    def pop_oldest(self):
        return self._commands.popitem(
            last=False
        )

    def pop_newest(self):
        return self._commands.popitem(
            last=True
        )

    def snapshot(self):
        return list(self._commands.items())


In [104]:
history = CommandHistory()

history.add("c1", "load data")
history.add("c2", "clean data")
history.add("c3", "train model")
history.pin("c3")

print(history.snapshot())

print("pop oldest:", history.pop_oldest())
print("pop newest:", history.pop_newest())
print("remaining:", history.snapshot())


[('c3', 'train model'), ('c1', 'load data'), ('c2', 'clean data')]
pop oldest: ('c3', 'train model')
pop newest: ('c2', 'clean data')
remaining: [('c1', 'load data')]


### Step 6 — Compare histories

Sometimes we care only about the command IDs/text.

Sometimes we care about their exact order as well.

We will support both.


In [105]:
def histories_equal(h1, h2, *, order_sensitive=True):
    if order_sensitive:
        return (
            h1.snapshot()
            == h2.snapshot()
        )

    return (
        dict(h1.snapshot())
        == dict(h2.snapshot())
    )


In [106]:
h1 = CommandHistory()
h1.add("a", "one")
h1.add("b", "two")
h1.add("c", "three")

h2 = CommandHistory()
h2.add("b", "two")
h2.add("c", "three")
h2.add("a", "one")

print(
    histories_equal(
        h1,
        h2,
        order_sensitive=False
    )
)

print(
    histories_equal(
        h1,
        h2,
        order_sensitive=True
    )
)


True
False


The first comparison is `True` because the histories contain the same command mappings.

The second is `False` because their command order is different.


# Final Discussion

Across these problems we repeatedly used the same few ideas in different settings.

A plain `dict` is often enough when we need:

- ordinary mapping behavior,
- stable insertion order,
- simple first/last inspection,
- occasional pop/reinsert operations.

`OrderedDict` becomes especially expressive when the application frequently needs:

- move-to-front,
- move-to-end,
- pop-first,
- pop-last,
- explicit order-aware behavior.

The most important question is therefore not:

> "Is a plain dictionary ordered?"

It is:

> "How much order manipulation is part of the actual workload?"


# Extra Practice Problems

These are left without solutions so you can extend the tutorial yourself.

1. Implement `rotate_right(d, n)` without calling `rotate_right_once` repeatedly.
2. Add a maximum capacity to `CommandHistory`.
3. Add `unpin(command_id)` semantics that move a command to the end.
4. Build a plain-dict version of `JobQueue`.
5. Compare the timing of `normalize_key_order` for 100, 1,000, and 10,000 keys.
6. Add a method to `MRURegistry` that returns the least-recent key without removing it.
7. Write a function that moves several selected keys to the front while preserving the selected order.
8. Build randomized tests for `rotate_right`.
9. Compare two `OrderedDict` instances directly, then compare their `dict(...)` conversions.
10. Design a benchmark where lookup is frequent but reordering is rare, and compare the two mapping types.
